# BizLens v2.2.12 - Process Mining

Analyze **business processes, workflows, and event sequences** using event logs.

**Now with optional pm4py integration for advanced process discovery.**

In [ ]:
import subprocess
import sys

# Install/upgrade BizLens with process-mining extras
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "bizlens", "plotly"])
print("✅ BizLens v2.2.13 + pm4py + Plotly installed/upgraded successfully!")

In [ ]:
import bizlens as bl
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Optional: Enable performance timing
# bl.set_profiling(True)

# ── Optional: Use a BizLens built-in dataset ─────────────────────────────
# event_log = bl.generate_hr_onboarding_event_log()  # HR onboarding — case_id, activity, timestamp, resource
# event_log = bl.generate_tech_support_event_log()  # tech support tickets — case_id, activity, timestamp, resource
#
# After loading, explore with:
# print(df.shape)         # rows × columns
# print(df.dtypes)        # column types
# print(df.head())        # first 5 rows
#
# ── Export / Save Results ─────────────────────────────────────────────────
# event_log.to_csv('process_log.csv', index=False)
# df.to_excel('output.xlsx', index=False)  # Save as Excel
# df.to_json('output.json', orient='records', indent=2)  # Save as JSON
# ── Polars alternative (optional — for users who prefer polars) ──────────────
# import polars as pl
# df_pl = pl.from_pandas(bl.load_dataset('titanic'))    # load → convert to polars
# df_pl = pl.read_csv('your_data.csv')                  # or read directly
# df_pl = pl.DataFrame(data)                             # or create from dict
#
# BizLens accepts both pandas and polars DataFrames transparently:
# bl.tables.frequency_table(df_pl['sex'])               # works with polars too
# bl.quality.data_profile(df_pl)                        # works with polars too
#
# Convert polars back to pandas when needed:
# df = df_pl.to_pandas()

## Dataset: Support Ticket Workflow (Event Log)

In [ ]:
np.random.seed(42)
events = []
base_time = datetime(2024, 1, 1)

for case_id in range(1, 101):
    activities = ['Create', 'Assign', 'Investigate', 'Resolve', 'Close']
    current_time = base_time + timedelta(days=case_id)
    for activity in activities:
        events.append({
            'case_id': f'TICKET-{case_id:03d}',
            'activity': activity,
            'timestamp': current_time,
            'agent': np.random.choice(['Alice', 'Bob', 'Charlie'])
        })
        current_time += timedelta(hours=np.random.gamma(2, 2))

event_log = pd.DataFrame(events)
print(f"Events: {len(event_log):,} | Cases: {event_log['case_id'].nunique():,}")

## 1. Case Metrics (BizLens)

In [ ]:
case_metrics = bl.process_mining.case_metrics(event_log)

## 2. Activity Analysis (BizLens)

In [ ]:
activity_metrics = bl.process_mining.activity_metrics(event_log)

## 3. Bottleneck Analysis (BizLens)

In [ ]:
bottlenecks = bl.process_mining.bottleneck_analysis(event_log)

## 4. Process Variants (BizLens)

In [ ]:
variants_table, variants_list = bl.process_mining.variant_discovery(event_log)

## 5. Resource Analysis (BizLens)

In [ ]:
resources = bl.process_mining.resource_analysis(event_log, resource_col='agent')

## Advanced: Process Mining with pm4py (Optional)

In [ ]:
try:
    import pm4py
    print("✅ pm4py is available - running advanced analysis")
    
    # Convert to pm4py event log format
    pm4py_log = pm4py.formatters.format_dataframe(event_log, case_id='case_id', activity_key='activity', timestamp_key='timestamp')
    
    # Process Discovery - Inductive Miner (recommended)
    net, initial_marking, final_marking = pm4py.discovery.discover_petri_net_inductive(pm4py_log)
    
    # Visualize Petri Net
    pm4py.vis.view_petri_net(net, initial_marking, final_marking, format='png')
    
    print("Petri Net generated successfully using pm4py")
    
except ImportError:
    print("⚠️  pm4py is not installed. Install with: pip install -q git+https://github.com/solutiongate-learn/bizlens.git")
except Exception as e:
    print(f"pm4py error: {e}")